<a href="https://colab.research.google.com/github/Fer448/Ejemplos_R/blob/main/sudokus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SUDOKUS

## GENERADOR PARA IMPRIMIR

In [ ]:
import random
import os
from datetime import datetime

class SudokuGenerator:
    def __init__(self, difficulty='medium'):
        """
        Inicializa el generador de sudokus.
        Niveles de dificultad: 'easy', 'medium', 'hard', 'expert'
        """
        self.difficulty = difficulty
        self.board = [[0 for _ in range(9)] for _ in range(9)]
        self.solution = None

        # Número de celdas vacías según dificultad
        self.empty_cells = {
            'easy': 30,
            'medium': 40,
            'hard': 50,
            'expert': 60
        }

    def generate_board(self):
        """Genera un tablero de sudoku completo y válido"""
        # Primero llenamos la diagonal de 3x3 con números aleatorios
        self.fill_diagonal()

        # Luego llenamos el resto del tablero
        self.fill_remaining(0, 3)

        # Guardamos la solución completa
        self.solution = [row[:] for row in self.board]

        # Eliminamos celdas según la dificultad
        self.remove_cells()

        return self.board

    def fill_diagonal(self):
        """Llena las 3 cajas diagonales de 3x3 con números aleatorios"""
        for i in range(0, 9, 3):
            self.fill_box(i, i)

    def fill_box(self, row, col):
        """Llena una caja de 3x3 con números aleatorios"""
        nums = list(range(1, 10))
        random.shuffle(nums)

        for i in range(3):
            for j in range(3):
                self.board[row + i][col + j] = nums.pop()

    def is_valid(self, row, col, num):
        """Verifica si es válido colocar un número en una posición específica"""
        # Verificar fila
        for x in range(9):
            if self.board[row][x] == num:
                return False

        # Verificar columna
        for x in range(9):
            if self.board[x][col] == num:
                return False

        # Verificar caja 3x3
        start_row = row - row % 3
        start_col = col - col % 3
        for i in range(3):
            for j in range(3):
                if self.board[start_row + i][start_col + j] == num:
                    return False

        return True

    def fill_remaining(self, row, col):
        """Llena el resto del tablero usando backtracking"""
        # Si llegamos al final del tablero, retornamos True
        if row == 8 and col == 9:
            return True

        # Si llegamos al final de una fila, pasamos a la siguiente
        if col == 9:
            row += 1
            col = 0

        # Si la celda ya está llena, pasamos a la siguiente
        if self.board[row][col] != 0:
            return self.fill_remaining(row, col + 1)

        # Probamos números del 1 al 9
        nums = list(range(1, 10))
        random.shuffle(nums)

        for num in nums:
            if self.is_valid(row, col, num):
                self.board[row][col] = num

                if self.fill_remaining(row, col + 1):
                    return True

                # Si no funciona, volvemos a 0
                self.board[row][col] = 0

        return False

    def remove_cells(self):
        """Elimina celdas según la dificultad seleccionada"""
        cells_to_remove = self.empty_cells[self.difficulty]
        removed = 0

        while removed < cells_to_remove:
            row = random.randint(0, 8)
            col = random.randint(0, 8)

            # Si la celda ya está vacía, continuamos
            if self.board[row][col] == 0:
                continue

            # Guardamos el valor antes de eliminarlo
            backup = self.board[row][col]
            self.board[row][col] = 0

            # Verificamos que todavía tenga solución única
            if self.has_unique_solution():
                removed += 1
            else:
                # Si no tiene solución única, restauramos el valor
                self.board[row][col] = backup

    def has_unique_solution(self):
        """Verifica si el tablero tiene solución única (simplificado)"""
        # Para simplificar, asumimos que después de eliminar las celdas
        # según los parámetros de dificultad, el sudoku tiene solución única
        return True

    def to_html(self, filename=None, include_solution=False):
        """Convierte el tablero de sudoku a formato HTML"""
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"sudoku_{self.difficulty}_{timestamp}.html"

        # Crear directorio si no existe
        if not os.path.exists("sudokus_html"):
            os.makedirs("sudokus_html")

        filepath = os.path.join("sudokus_html", filename)

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(self._generate_html_content(include_solution))

        print(f"Sudoku guardado en: {filepath}")
        return filepath

    def _generate_html_content(self, include_solution):
        """Genera el contenido HTML completo"""
        html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sudoku {self.difficulty.capitalize()}</title>
    <style>
        * {{
            box-sizing: border-box;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }}

        body {{
            background-color: #f0f2f5;
            color: #333;
            margin: 0;
            padding: 20px;
            display: flex;
            flex-direction: column;
            align-items: center;
            min-height: 100vh;
        }}

        .container {{
            max-width: 1200px;
            width: 100%;
            margin: 0 auto;
        }}

        header {{
            text-align: center;
            margin-bottom: 30px;
            padding: 20px;
            background-color: #4a6fa5;
            color: white;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            width: 100%;
        }}

        h1 {{
            margin: 0 0 10px 0;
            font-size: 2.5em;
        }}

        .difficulty {{
            font-size: 1.2em;
            font-weight: bold;
            padding: 5px 15px;
            border-radius: 20px;
            display: inline-block;
            margin-top: 10px;
        }}

        .easy {{
            background-color: #4CAF50;
        }}

        .medium {{
            background-color: #FF9800;
        }}

        .hard {{
            background-color: #F44336;
        }}

        .expert {{
            background-color: #9C27B0;
        }}

        .content {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            gap: 40px;
            margin-bottom: 30px;
        }}

        .sudoku-container {{
            flex: 1;
            min-width: 300px;
            max-width: 500px;
        }}

        .sudoku-board {{
            display: grid;
            grid-template-columns: repeat(9, 1fr);
            grid-template-rows: repeat(9, 1fr);
            gap: 0;
            border: 3px solid #333;
            background-color: white;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }}

        .sudoku-cell {{
            aspect-ratio: 1;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 1.5em;
            font-weight: bold;
            border: 1px solid #ccc;
            user-select: none;
        }}

        /* Bordes más gruesos para las cajas 3x3 */
        .sudoku-cell:nth-child(3n) {{
            border-right: 2px solid #333;
        }}

        .sudoku-cell:nth-child(9n) {{
            border-right: none;
        }}

        .sudoku-cell:nth-child(n+19):nth-child(-n+27),
        .sudoku-cell:nth-child(n+46):nth-child(-n+54) {{
            border-bottom: 2px solid #333;
        }}

        /* Colores para las celdas vacías y con números */
        .empty {{
            color: transparent;
            background-color: #f9f9f9;
        }}

        .given {{
            color: #2c3e50;
            background-color: #e8f4f8;
        }}

        .solution-cell {{
            color: #27ae60;
            background-color: #eafaf1;
        }}

        .instructions {{
            flex: 1;
            min-width: 300px;
            max-width: 500px;
            background-color: white;
            padding: 25px;
            border-radius: 10px;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }}

        h2 {{
            color: #4a6fa5;
            margin-top: 0;
        }}

        ul {{
            line-height: 1.6;
            padding-left: 20px;
        }}

        .rules {{
            margin-top: 25px;
            padding-top: 20px;
            border-top: 1px solid #eee;
        }}

        footer {{
            text-align: center;
            margin-top: 20px;
            padding: 15px;
            color: #666;
            font-size: 0.9em;
            width: 100%;
            border-top: 1px solid #ddd;
        }}

        .print-button {{
            background-color: #4a6fa5;
            color: white;
            border: none;
            padding: 12px 25px;
            font-size: 1em;
            border-radius: 5px;
            cursor: pointer;
            margin: 20px 0;
            transition: background-color 0.3s;
        }}

        .print-button:hover {{
            background-color: #3a5a80;
        }}

        @media print {{
            .print-button, footer {{
                display: none;
            }}

            body {{
                padding: 0;
            }}

            .sudoku-board {{
                box-shadow: none;
            }}
        }}

        @media (max-width: 768px) {{
            .content {{
                flex-direction: column;
                align-items: center;
            }}

            .sudoku-container, .instructions {{
                max-width: 100%;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>SUDOKU</h1>
            <div class="difficulty {self.difficulty}">{self.difficulty.upper()}</div>
            <p>Generado el {datetime.now().strftime('%d/%m/%Y a las %H:%M')}</p>
        </header>

        <button class="print-button" onclick="window.print()">🖨️ Imprimir Sudoku</button>

        <div class="content">
            <div class="sudoku-container">
                <h2>Sudoku para Resolver</h2>
                <div class="sudoku-board">
"""
        # Generar el tablero para resolver
        for row in range(9):
            for col in range(9):
                value = self.board[row][col]
                cell_class = "given" if value != 0 else "empty"
                cell_content = value if value != 0 else "0"

                html_content += f'<div class="sudoku-cell {cell_class}">{cell_content}</div>\n'

        html_content += """                </div>
            </div>

            <div class="instructions">
                <h2>¿Cómo jugar?</h2>
                <ul>
                    <li>El objetivo es llenar la cuadrícula de 9×9 con dígitos del 1 al 9</li>
                    <li>Cada fila debe contener todos los dígitos del 1 al 9 sin repetir</li>
                    <li>Cada columna debe contener todos los dígitos del 1 al 9 sin repetir</li>
                    <li>Cada caja de 3×3 debe contener todos los dígitos del 1 al 9 sin repetir</li>
                    <li>Los números en gris ya están colocados y no pueden cambiarse</li>
                </ul>

                <div class="rules">
                    <h2>Consejos para resolver</h2>
                    <ul>
                        <li>Comienza por los números que ya están colocados</li>
                        <li>Busca filas, columnas o cajas con muchos números ya colocados</li>
                        <li>Usa lógica deductiva, no adivines</li>
                        <li>Si te atascas, intenta con otro enfoque</li>
                    </ul>
                </div>
"""

        # Incluir solución si se solicita
        if include_solution and self.solution:
            html_content += """
                <div class="rules">
                    <h2>Solución</h2>
                    <div class="sudoku-board">
"""
            # Generar el tablero con la solución
            for row in range(9):
                for col in range(9):
                    original_value = self.board[row][col]
                    solution_value = self.solution[row][col]

                    if original_value == 0:
                        # Celda que estaba vacía, mostrar solución en verde
                        html_content += f'<div class="sudoku-cell solution-cell">{solution_value}</div>\n'
                    else:
                        # Celda que ya tenía número, mostrarlo normal
                        html_content += f'<div class="sudoku-cell given">{solution_value}</div>\n'

            html_content += """                    </div>
                </div>
"""

        html_content += f"""
            </div>
        </div>

        <footer>
            <p>Sudoku generado automáticamente con Python • Dificultad: {self.difficulty.capitalize()} • {self.empty_cells[self.difficulty]} celdas vacías</p>
        </footer>
    </div>

    <script>
        // Añadir interactividad básica para imprimir
        document.addEventListener('DOMContentLoaded', function() {{
            console.log('Sudoku cargado correctamente');
        }});
    </script>
</body>
</html>"""

        return html_content


def generar_sudokus(cantidad=1, dificultad='medium', incluir_solucion=False):
    """
    Genera múltiples sudokus y los guarda como archivos HTML.

    Parámetros:
    - cantidad: número de sudokus a generar
    - dificultad: 'easy', 'medium', 'hard', 'expert'
    - incluir_solucion: si True, incluye la solución en el HTML
    """
    print(f"Generando {cantidad} sudoku(s) de dificultad {dificultad}...")

    for i in range(cantidad):
        generator = SudokuGenerator(difficulty=dificultad)
        sudoku = generator.generate_board()

        filename = f"sudoku_{dificultad}_{i+1}.html"
        generator.to_html(filename=filename, include_solution=incluir_solucion)

        print(f"  Sudoku {i+1} generado con éxito")

    print(f"\n¡Todos los sudokus han sido guardados en la carpeta 'sudokus_html'!")


def menu_interactivo():
    """Muestra un menú interactivo para generar sudokus"""
    print("=" * 50)
    print("GENERADOR DE SUDOKUS EN HTML")
    print("=" * 50)

    while True:
        print("\nOpciones:")
        print("1. Generar un sudoku fácil")
        print("2. Generar un sudoku medio")
        print("3. Generar un sudoku difícil")
        print("4. Generar un sudoku experto")
        print("5. Generar múltiples sudokus")
        print("6. Generar sudokus de todas las dificultades")
        print("7. Salir")

        opcion = input("\nSelecciona una opción (1-7): ").strip()

        if opcion == "1":
            generar_sudokus(1, 'easy')
        elif opcion == "2":
            generar_sudokus(1, 'medium')
        elif opcion == "3":
            generar_sudokus(1, 'hard')
        elif opcion == "4":
            generar_sudokus(1, 'expert')
        elif opcion == "5":
            try:
                cantidad = int(input("¿Cuántos sudokus quieres generar? "))
                dificultad = input("¿Dificultad? (easy/medium/hard/expert): ").lower()
                if dificultad not in ['easy', 'medium', 'hard', 'expert']:
                    print("Dificultad no válida. Usando 'medium' por defecto.")
                    dificultad = 'medium'

                solucion = input("¿Incluir solución? (s/n): ").lower() == 's'
                generar_suduks(cantidad, dificultad, solucion)
            except ValueError:
                print("Por favor, introduce un número válido.")
        elif opcion == "6":
            print("\nGenerando sudokus de todas las dificultades...")
            for dificultad in ['easy', 'medium', 'hard', 'expert']:
                generar_sudokus(1, dificultad)
            print("\n¡Se han generado sudokus de todas las dificultades!")
        elif opcion == "7":
            print("¡Hasta luego!")
            break
        else:
            print("Opción no válida. Por favor, selecciona una opción del 1 al 7.")


if __name__ == "__main__":
    # Ejemplo de uso directo
    print("Generador de Sudokus en HTML")
    print("-" * 30)

    # Generar un sudoku medio por defecto
    generator = SudokuGenerator(difficulty='medium')
    sudoku = generator.generate_board()
    generator.to_html(include_solution=True)

    # Para usar el menú interactivo, descomenta la siguiente línea:
    # menu_interactivo()

    # Para generar múltiples sudokus directamente:
    # generar_sudokus(3, 'medium', incluir_solucion=True)

## GENERADOR PARA HTML INTERACTIVO

In [ ]:
import random
import json
import os
from datetime import datetime

class SudokuGenerator:
    def __init__(self, difficulty='medium'):
        """
        Inicializa el generador de sudokus con diferentes niveles de dificultad.

        Args:
            difficulty (str): Nivel de dificultad ('easy', 'medium', 'hard', 'expert')
        """
        self.difficulty = difficulty
        self.size = 9
        self.board = [[0 for _ in range(self.size)] for _ in range(self.size)]
        self.solution = None

        # Número de celdas a eliminar según la dificultad
        self.cells_to_remove = {
            'easy': 30,
            'medium': 40,
            'hard': 50,
            'expert': 55
        }

    def is_valid(self, board, row, col, num):
        """Verifica si es válido colocar un número en una posición específica."""
        # Verifica la fila
        for x in range(self.size):
            if board[row][x] == num:
                return False

        # Verifica la columna
        for x in range(self.size):
            if board[x][col] == num:
                return False

        # Verifica el cuadro 3x3
        start_row, start_col = row - row % 3, col - col % 3
        for i in range(3):
            for j in range(3):
                if board[i + start_row][j + start_col] == num:
                    return False

        return True

    def fill_diagonal(self):
        """Llena los bloques diagonales 3x3 con números aleatorios."""
        for i in range(0, self.size, 3):
            self.fill_block(i, i)

    def fill_block(self, row, col):
        """Llena un bloque 3x3 con números aleatorios."""
        nums = list(range(1, 10))
        random.shuffle(nums)

        for i in range(3):
            for j in range(3):
                self.board[row + i][col + j] = nums.pop()

    def fill_remaining(self, row, col):
        """Llena el resto del tablero usando backtracking."""
        # Si hemos llegado al final del tablero, retornamos True
        if row == self.size - 1 and col == self.size:
            return True

        # Si llegamos al final de la fila, pasamos a la siguiente
        if col == self.size:
            row += 1
            col = 0

        # Si la celda ya está llena, pasamos a la siguiente
        if self.board[row][col] != 0:
            return self.fill_remaining(row, col + 1)

        # Intentamos colocar números del 1 al 9
        for num in range(1, 10):
            if self.is_valid(self.board, row, col, num):
                self.board[row][col] = num
                if self.fill_remaining(row, col + 1):
                    return True
                self.board[row][col] = 0

        return False

    def generate_full_sudoku(self):
        """Genera un sudoku completo válido."""
        self.board = [[0 for _ in range(self.size)] for _ in range(self.size)]
        self.fill_diagonal()
        self.fill_remaining(0, 3)
        return self.board

    def remove_numbers(self, board):
        """Elimina números del tablero según la dificultad."""
        puzzle = [row[:] for row in board]
        cells = self.cells_to_remove.get(self.difficulty, 40)

        attempts = cells
        while attempts > 0:
            row = random.randint(0, 8)
            col = random.randint(0, 8)

            # Si la celda ya está vacía, continuamos
            if puzzle[row][col] == 0:
                continue

            # Guardamos el valor original
            original = puzzle[row][col]
            puzzle[row][col] = 0

            # Verificamos que el puzzle siga teniendo solución única
            temp_puzzle = [row[:] for row in puzzle]
            if not self.has_unique_solution(temp_puzzle):
                puzzle[row][col] = original
                continue

            attempts -= 1

        return puzzle

    def has_unique_solution(self, board):
        """Verifica si el sudoku tiene solución única."""
        # Implementación simplificada para verificar unicidad
        return self.solve_sudoku(board, count_solutions=True) == 1

    def solve_sudoku(self, board, count_solutions=False):
        """Resuelve el sudoku y cuenta soluciones si es necesario."""
        empty = self.find_empty(board)

        if not empty:
            return 1  # Solución encontrada

        row, col = empty
        solutions = 0

        for num in range(1, 10):
            if self.is_valid(board, row, col, num):
                board[row][col] = num

                if count_solutions:
                    solutions += self.solve_sudoku(board, count_solutions)
                    if solutions > 1:  # Si encontramos más de una solución, paramos
                        board[row][col] = 0
                        return solutions
                else:
                    if self.solve_sudoku(board, count_solutions):
                        return True

                board[row][col] = 0

        return solutions if count_solutions else False

    def find_empty(self, board):
        """Encuentra una celda vacía en el tablero."""
        for i in range(self.size):
            for j in range(self.size):
                if board[i][j] == 0:
                    return (i, j)
        return None

    def generate(self):
        """Genera un sudoku con su solución."""
        full_sudoku = self.generate_full_sudoku()
        puzzle = self.remove_numbers(full_sudoku)
        self.solution = full_sudoku
        return puzzle, full_sudoku


class HTMLSudokuExporter:
    def __init__(self):
        """Inicializa el exportador HTML para sudokus."""
        # Para evitar problemas con las llaves en el CSS, usaremos formato diferente
        self.html_template = """<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sudoku Interactivo</title>
    <style>
        * {{
            box-sizing: border-box;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }}

        body {{
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            min-height: 100vh;
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%);
            color: #333;
        }}

        .container {{
            background-color: white;
            border-radius: 15px;
            box-shadow: 0 10px 30px rgba(0, 0, 0, 0.2);
            padding: 30px;
            max-width: 800px;
            width: 100%;
        }}

        h1 {{
            text-align: center;
            color: #2c3e50;
            margin-top: 0;
            margin-bottom: 10px;
            font-size: 2.5rem;
        }}

        .subtitle {{
            text-align: center;
            color: #7f8c8d;
            margin-bottom: 30px;
            font-size: 1.1rem;
        }}

        .difficulty-badge {{
            display: inline-block;
            padding: 5px 15px;
            border-radius: 20px;
            font-weight: bold;
            margin-left: 10px;
        }}

        .easy {{ background-color: #2ecc71; color: white; }}
        .medium {{ background-color: #f39c12; color: white; }}
        .hard {{ background-color: #e74c3c; color: white; }}
        .expert {{ background-color: #8e44ad; color: white; }}

        .game-container {{
            display: flex;
            flex-wrap: wrap;
            gap: 30px;
            justify-content: center;
            margin-bottom: 30px;
        }}

        .sudoku-board {{
            border: 3px solid #2c3e50;
            border-collapse: collapse;
            margin: 0 auto;
        }}

        .sudoku-cell {{
            width: 50px;
            height: 50px;
            text-align: center;
            font-size: 1.5rem;
            font-weight: bold;
            border: 1px solid #bdc3c7;
            background-color: white;
            transition: all 0.2s;
        }}

        .sudoku-cell:focus {{
            outline: 2px solid #3498db;
            background-color: #f0f8ff;
        }}

        .sudoku-cell.given {{
            background-color: #f8f9fa;
            font-weight: bold;
            color: #2c3e50;
        }}

        .sudoku-cell.user-input {{
            color: #3498db;
        }}

        .sudoku-cell.error {{
            color: #e74c3c;
            background-color: #ffeaea;
        }}

        .sudoku-cell.correct {{
            color: #27ae60;
            background-color: #eafaf1;
        }}

        /* Bordes más gruesos para los bloques 3x3 */
        .sudoku-cell:nth-child(3n) {{
            border-right: 3px solid #2c3e50;
        }}

        .sudoku-cell:nth-child(3n+1) {{
            border-left: 3px solid #2c3e50;
        }}

        tr:nth-child(3n) .sudoku-cell {{
            border-bottom: 3px solid #2c3e50;
        }}

        tr:nth-child(3n+1) .sudoku-cell {{
            border-top: 3px solid #2c3e50;
        }}

        .controls {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            gap: 15px;
            margin: 25px 0;
        }}

        button {{
            padding: 12px 25px;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 1rem;
            font-weight: bold;
            transition: all 0.3s;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 8px;
        }}

        button:hover {{
            transform: translateY(-3px);
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }}

        .check-btn {{
            background-color: #2ecc71;
            color: white;
        }}

        .solve-btn {{
            background-color: #3498db;
            color: white;
        }}

        .new-btn {{
            background-color: #9b59b6;
            color: white;
        }}

        .hint-btn {{
            background-color: #f1c40f;
            color: white;
        }}

        .clear-btn {{
            background-color: #e74c3c;
            color: white;
        }}

        .info-panel {{
            background-color: #f8f9fa;
            border-radius: 10px;
            padding: 20px;
            margin-top: 20px;
        }}

        .info-panel h3 {{
            margin-top: 0;
            color: #2c3e50;
        }}

        .stats {{
            display: flex;
            justify-content: space-between;
            flex-wrap: wrap;
            gap: 15px;
        }}

        .stat {{
            text-align: center;
            flex: 1;
            min-width: 120px;
        }}

        .stat-value {{
            font-size: 2rem;
            font-weight: bold;
            color: #3498db;
        }}

        .stat-label {{
            font-size: 0.9rem;
            color: #7f8c8d;
        }}

        .timer {{
            font-size: 1.8rem;
            font-weight: bold;
            text-align: center;
            color: #2c3e50;
            margin: 20px 0;
        }}

        .message {{
            text-align: center;
            padding: 15px;
            border-radius: 8px;
            margin: 20px 0;
            font-weight: bold;
            display: none;
        }}

        .success {{
            background-color: #d4edda;
            color: #155724;
            border: 1px solid #c3e6cb;
        }}

        .error {{
            background-color: #f8d7da;
            color: #721c24;
            border: 1px solid #f5c6cb;
        }}

        .hint {{
            background-color: #fff3cd;
            color: #856404;
            border: 1px solid #ffeaa7;
        }}

        footer {{
            text-align: center;
            margin-top: 30px;
            color: white;
            font-size: 0.9rem;
        }}

        @media (max-width: 768px) {{
            .sudoku-cell {{
                width: 40px;
                height: 40px;
                font-size: 1.2rem;
            }}

            .controls {{
                flex-direction: column;
                align-items: center;
            }}

            button {{
                width: 100%;
                max-width: 300px;
            }}
        }}

        @media (max-width: 480px) {{
            .sudoku-cell {{
                width: 35px;
                height: 35px;
                font-size: 1rem;
            }}

            .container {{
                padding: 15px;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>Sudoku Interactivo</h1>
        <div class="subtitle">
            Dificultad: <span id="difficulty" class="difficulty-badge {difficulty_class}">{difficulty}</span>
            | Generado: {date}
        </div>

        <div class="timer" id="timer">00:00:00</div>

        <div class="message" id="message"></div>

        <div class="game-container">
            <table class="sudoku-board" id="sudoku-board">
                {board_html}
            </table>
        </div>

        <div class="controls">
            <button class="check-btn" onclick="checkSolution()">
                <span>✓</span> Verificar
            </button>
            <button class="hint-btn" onclick="giveHint()">
                <span>?</span> Pista
            </button>
            <button class="solve-btn" onclick="solvePuzzle()">
                <span>≡</span> Resolver
            </button>
            <button class="clear-btn" onclick="clearUserInput()">
                <span>×</span> Limpiar
            </button>
            <button class="new-btn" onclick="newPuzzle()">
                <span>↻</span> Nuevo Sudoku
            </button>
        </div>

        <div class="info-panel">
            <h3>Instrucciones</h3>
            <p>Llena el tablero de sudoku con números del 1 al 9. Cada fila, columna y bloque 3x3 debe contener todos los números del 1 al 9 sin repetirse.</p>

            <div class="stats">
                <div class="stat">
                    <div class="stat-value" id="hints-used">0</div>
                    <div class="stat-label">Pistas usadas</div>
                </div>
                <div class="stat">
                    <div class="stat-value" id="errors-found">0</div>
                    <div class="stat-label">Errores encontrados</div>
                </div>
                <div class="stat">
                    <div class="stat-value" id="cells-filled">0</div>
                    <div class="stat-label">Celdas llenadas</div>
                </div>
                <div class="stat">
                    <div class="stat-value" id="completion">0%</div>
                    <div class="stat-label">Completado</div>
                </div>
            </div>
        </div>
    </div>

    <footer>
        Sudoku generado con Python | Creado el {date}
    </footer>

    <script>
        // Datos del sudoku
        const initialBoard = {initial_board};
        const solution = {solution};
        let currentBoard = JSON.parse(JSON.stringify(initialBoard));
        let startTime = new Date();
        let timerInterval;
        let hintsUsed = 0;
        let errorsFound = 0;

        // Función para comparar valores (maneja números como strings o enteros)
        function compareValues(userValue, correctValue) {{
            // Convierte ambos a números para comparar
            const userNum = parseInt(userValue);
            const correctNum = parseInt(correctValue);

            // Si alguno no es un número válido, retorna false
            if (isNaN(userNum) || isNaN(correctNum)) {{
                return false;
            }}

            return userNum === correctNum;
        }}

        // Inicializar el tablero
        function initializeBoard() {{
            const table = document.getElementById('sudoku-board');
            table.innerHTML = '';

            for (let i = 0; i < 9; i++) {{
                const row = document.createElement('tr');

                for (let j = 0; j < 9; j++) {{
                    const cell = document.createElement('td');
                    const input = document.createElement('input');

                    input.type = 'text';
                    input.maxLength = 1;
                    input.className = 'sudoku-cell';
                    input.dataset.row = i;
                    input.dataset.col = j;

                    // Si es un número dado (no editable)
                    if (initialBoard[i][j] !== 0) {{
                        input.value = initialBoard[i][j];
                        input.readOnly = true;
                        input.classList.add('given');
                    }} else {{
                        // Si el usuario ya ingresó un valor
                        if (currentBoard[i][j] !== 0) {{
                            input.value = currentBoard[i][j];
                            input.classList.add('user-input');

                            // Verificar si es correcto
                            if (!compareValues(input.value, solution[i][j])) {{
                                input.classList.add('error');
                            }} else {{
                                input.classList.add('correct');
                            }}
                        }}

                        // Eventos para entrada del usuario
                        input.addEventListener('input', function(e) {{
                            const value = e.target.value;
                            const row = parseInt(e.target.dataset.row);
                            const col = parseInt(e.target.dataset.col);

                            // Solo permitir números del 1 al 9
                            if (value === '' || (/^[1-9]$/.test(value))) {{
                                currentBoard[row][col] = value === '' ? 0 : parseInt(value);

                                // Limpiar clases anteriores
                                e.target.classList.remove('error', 'correct', 'user-input');

                                if (value !== '') {{
                                    e.target.classList.add('user-input');

                                    // Verificar si es correcto
                                    if (compareValues(value, solution[row][col])) {{
                                        e.target.classList.add('correct');
                                    }} else {{
                                        e.target.classList.add('error');
                                        errorsFound++;
                                        document.getElementById('errors-found').textContent = errorsFound;
                                    }}
                                }}

                                updateStats();
                            }} else {{
                                e.target.value = '';
                            }}
                        }});

                        input.addEventListener('focus', function() {{
                            this.style.backgroundColor = '#f0f8ff';
                        }});

                        input.addEventListener('blur', function() {{
                            if (!this.classList.contains('error') && !this.classList.contains('given')) {{
                                this.style.backgroundColor = 'white';
                            }}
                        }});
                    }}

                    cell.appendChild(input);
                    row.appendChild(cell);
                }}

                table.appendChild(row);
            }}

            startTimer();
            updateStats();
        }}

        // Actualizar estadísticas
        function updateStats() {{
            let cellsFilled = 0;
            let totalCells = 0;

            for (let i = 0; i < 9; i++) {{
                for (let j = 0; j < 9; j++) {{
                    if (initialBoard[i][j] === 0) {{
                        totalCells++;
                        if (currentBoard[i][j] !== 0) {{
                            cellsFilled++;
                        }}
                    }}
                }}
            }}

            const completion = totalCells > 0 ? Math.round((cellsFilled / totalCells) * 100) : 100;

            document.getElementById('cells-filled').textContent = cellsFilled;
            document.getElementById('completion').textContent = completion + '%';
            document.getElementById('hints-used').textContent = hintsUsed;
        }}

        // Iniciar temporizador
        function startTimer() {{
            clearInterval(timerInterval);
            startTime = new Date();

            timerInterval = setInterval(() => {{
                const now = new Date();
                const diff = new Date(now - startTime);

                const hours = diff.getUTCHours().toString().padStart(2, '0');
                const minutes = diff.getUTCMinutes().toString().padStart(2, '0');
                const seconds = diff.getUTCSeconds().toString().padStart(2, '0');

                document.getElementById('timer').textContent = `${{hours}}:${{minutes}}:${{seconds}}`;
            }}, 1000);
        }}

        // Verificar solución
        function checkSolution() {{
            let isCorrect = true;
            let isComplete = true;
            const inputs = document.querySelectorAll('.sudoku-cell:not(.given)');

            inputs.forEach(input => {{
                const row = parseInt(input.dataset.row);
                const col = parseInt(input.dataset.col);
                const userValue = input.value;

                // Limpiar clases anteriores
                input.classList.remove('error', 'correct');

                if (userValue === '') {{
                    isComplete = false;
                    input.classList.remove('user-input');
                }} else {{
                    input.classList.add('user-input');

                    if (compareValues(userValue, solution[row][col])) {{
                        input.classList.add('correct');
                    }} else {{
                        input.classList.add('error');
                        isCorrect = false;
                    }}
                }}
            }});

            // Verificar si todas las celdas están llenas
            for (let i = 0; i < 9; i++) {{
                for (let j = 0; j < 9; j++) {{
                    if (initialBoard[i][j] === 0 && currentBoard[i][j] === 0) {{
                        isComplete = false;
                        break;
                    }}
                }}
                if (!isComplete) break;
            }}

            const message = document.getElementById('message');

            if (isCorrect && isComplete) {{
                message.textContent = '¡Felicidades! Has resuelto el sudoku correctamente.';
                message.className = 'message success';
                clearInterval(timerInterval);
            }} else if (isCorrect && !isComplete) {{
                message.textContent = 'Hasta ahora todas las respuestas son correctas, pero el sudoku no está completo.';
                message.className = 'message hint';
            }} else {{
                message.textContent = 'Hay errores en tu solución. Revisa las celdas marcadas en rojo.';
                message.className = 'message error';
            }}

            message.style.display = 'block';
            setTimeout(() => {{
                message.style.display = 'none';
            }}, 5000);
        }}

        // Dar una pista
        function giveHint() {{
            // Encontrar una celda vacía aleatoria
            let emptyCells = [];

            for (let i = 0; i < 9; i++) {{
                for (let j = 0; j < 9; j++) {{
                    if (currentBoard[i][j] === 0) {{
                        emptyCells.push({{row: i, col: j}});
                    }}
                }}
            }}

            if (emptyCells.length === 0) {{
                showMessage('¡Ya has completado el sudoku!', 'hint');
                return;
            }}

            const randomCell = emptyCells[Math.floor(Math.random() * emptyCells.length)];
            const row = randomCell.row;
            const col = randomCell.col;

            // Actualizar el tablero actual
            currentBoard[row][col] = solution[row][col];

            // Actualizar la interfaz
            const input = document.querySelector(`input[data-row="${{row}}"][data-col="${{col}}"]`);
            input.value = solution[row][col];
            input.classList.add('given', 'correct');
            input.readOnly = true;

            hintsUsed++;
            updateStats();

            showMessage(`Pista: La celda (${{row + 1}}, ${{col + 1}}) es ${{solution[row][col]}}`, 'hint');
        }}

        // Resolver el puzzle completo
        function solvePuzzle() {{
            if (confirm('¿Estás seguro de que quieres ver la solución completa? Esto reiniciará tu progreso.')) {{
                currentBoard = JSON.parse(JSON.stringify(solution));
                initializeBoard();
                clearInterval(timerInterval);
                showMessage('Sudoku resuelto automáticamente.', 'success');
            }}
        }}

        // Limpiar entradas del usuario
        function clearUserInput() {{
            if (confirm('¿Estás seguro de que quieres limpiar todas tus respuestas?')) {{
                currentBoard = JSON.parse(JSON.stringify(initialBoard));
                initializeBoard();
                hintsUsed = 0;
                errorsFound = 0;
                document.getElementById('errors-found').textContent = '0';
                showMessage('Todas tus respuestas han sido eliminadas.', 'hint');
            }}
        }}

        // Generar un nuevo puzzle
        function newPuzzle() {{
            if (confirm('¿Estás seguro de que quieres generar un nuevo sudoku? Se perderá tu progreso actual.')) {{
                // En una implementación real, aquí cargaríamos un nuevo sudoku del servidor
                // Por ahora, recargamos la página
                location.reload();
            }}
        }}

        // Mostrar mensaje
        function showMessage(text, type) {{
            const message = document.getElementById('message');
            message.textContent = text;
            message.className = `message ${{type}}`;
            message.style.display = 'block';

            setTimeout(() => {{
                message.style.display = 'none';
            }}, 5000);
        }}

        // Inicializar cuando se carga la página
        document.addEventListener('DOMContentLoaded', function() {{
            initializeBoard();

            // Permitir navegación con teclado
            document.addEventListener('keydown', function(e) {{
                if (e.target.classList.contains('sudoku-cell') && !e.target.readOnly) {{
                    const row = parseInt(e.target.dataset.row);
                    const col = parseInt(e.target.dataset.col);

                    let nextRow = row;
                    let nextCol = col;

                    switch(e.key) {{
                        case 'ArrowUp':
                            nextRow = row > 0 ? row - 1 : 8;
                            break;
                        case 'ArrowDown':
                            nextRow = row < 8 ? row + 1 : 0;
                            break;
                        case 'ArrowLeft':
                            nextCol = col > 0 ? col - 1 : 8;
                            break;
                        case 'ArrowRight':
                            nextCol = col < 8 ? col + 1 : 0;
                            break;
                        default:
                            return;
                    }}

                    const nextInput = document.querySelector(
                        `input[data-row="${{nextRow}}"][data-col="${{nextCol}}"]`
                    );

                    if (nextInput && !nextInput.readOnly) {{
                        nextInput.focus();
                        e.preventDefault();
                    }}
                }}
            }});
        }});
    </script>
</body>
</html>"""

    def generate_board_html(self, board, initial_board):
        """Genera el HTML para el tablero de sudoku."""
        html = ""
        for i in range(9):
            html += "<tr>"
            for j in range(9):
                value = board[i][j] if board[i][j] != 0 else ""
                readonly = "readonly" if initial_board[i][j] != 0 else ""
                cell_class = "given" if initial_board[i][j] != 0 else ""

                html += f'<td><input type="text" class="sudoku-cell {cell_class}" maxlength="1" value="{value}" {readonly} data-row="{i}" data-col="{j}"></td>'
            html += "</tr>"
        return html

    def get_difficulty_class(self, difficulty):
        """Obtiene la clase CSS para el nivel de dificultad."""
        classes = {
            'easy': 'easy',
            'medium': 'medium',
            'hard': 'hard',
            'expert': 'expert'
        }
        return classes.get(difficulty, 'medium')

    def export(self, puzzle, solution, difficulty='medium', filename='sudoku_interactivo.html'):
        """Exporta el sudoku a un archivo HTML interactivo."""
        # Preparar los datos para JavaScript
        initial_board_js = json.dumps(puzzle)
        solution_js = json.dumps(solution)

        # Generar HTML del tablero
        board_html = self.generate_board_html(puzzle, puzzle)

        # Obtener fecha actual
        date = datetime.now().strftime("%d/%m/%Y %H:%M")

        # Obtener clase de dificultad
        difficulty_class = self.get_difficulty_class(difficulty)

        # Reemplazar marcadores de posición en la plantilla
        html_content = self.html_template.format(
            difficulty=difficulty.upper(),
            difficulty_class=difficulty_class,
            date=date,
            board_html=board_html,
            initial_board=initial_board_js,
            solution=solution_js
        )

        # Guardar el archivo HTML
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)

        print(f"Sudoku interactivo guardado como '{filename}'")
        print(f"Dificultad: {difficulty}")
        print(f"Archivo listo para abrir en cualquier navegador web.")
        return os.path.abspath(filename)


def main():
    """Función principal para generar sudokus y exportarlos a HTML."""
    print("=" * 60)
    print("GENERADOR DE SUDOKUS INTERACTIVOS")
    print("=" * 60)

    # Solicitar nivel de dificultad al usuario
    print("\nSelecciona el nivel de dificultad:")
    print("1. Fácil")
    print("2. Medio")
    print("3. Difícil")
    print("4. Experto")

    choice = input("\nIngresa tu opción (1-4, predeterminado: 2): ").strip()

    difficulty_map = {
        '1': 'easy',
        '2': 'medium',
        '3': 'hard',
        '4': 'expert'
    }

    difficulty = difficulty_map.get(choice, 'medium')

    # Solicitar nombre de archivo
    default_filename = f"sudoku_{difficulty}.html"
    filename = input(f"\nNombre del archivo HTML (predeterminado: {default_filename}): ").strip()
    if not filename:
        filename = default_filename

    if not filename.endswith('.html'):
        filename += '.html'

    # Generar sudoku
    print(f"\nGenerando sudoku ({difficulty})...")
    generator = SudokuGenerator(difficulty)
    puzzle, solution = generator.generate()

    # Mostrar sudoku en consola
    print("\nSudoku generado (0 = vacío):")
    print("-" * 25)
    for i in range(9):
        row = "| "
        for j in range(9):
            if puzzle[i][j] == 0:
                row += ". "
            else:
                row += f"{puzzle[i][j]} "

            if (j + 1) % 3 == 0:
                row += "| "

        print(row)
        if (i + 1) % 3 == 0:
            print("-" * 25)

    # Mostrar solución en consola (opcional)
    show_solution = input("\n¿Deseas ver la solución en consola? (s/n): ").strip().lower()
    if show_solution == 's':
        print("\nSolución del sudoku:")
        print("-" * 25)
        for i in range(9):
            row = "| "
            for j in range(9):
                row += f"{solution[i][j]} "
                if (j + 1) % 3 == 0:
                    row += "| "
            print(row)
            if (i + 1) % 3 == 0:
                print("-" * 25)

    # Exportar a HTML
    print(f"\nExportando a HTML interactivo...")
    exporter = HTMLSudokuExporter()
    filepath = exporter.export(puzzle, solution, difficulty, filename)

    # Mostrar información adicional
    print("\n" + "=" * 60)
    print("INSTRUCCIONES PARA USAR EL SUDOKU INTERACTIVO:")
    print("=" * 60)
    print("1. Abre el archivo HTML en cualquier navegador web")
    print("2. Completa las celdas vacías con números del 1 al 9")
    print("3. Usa los botones para:")
    print("   - Verificar tu solución")
    print("   - Obtener pistas")
    print("   - Ver la solución completa")
    print("   - Limpiar tus respuestas")
    print("   - Generar un nuevo sudoku")
    print("\n¡Disfruta del juego!")

    # Preguntar si desea abrir el archivo
    open_file = input("\n¿Deseas abrir el archivo en tu navegador? (s/n): ").strip().lower()
    if open_file == 's':
        try:
            import webbrowser
            webbrowser.open(f'file://{filepath}')
            print("Abriendo en el navegador...")
        except Exception as e:
            print(f"No se pudo abrir automáticamente: {e}")
            print(f"Por favor, abre '{filename}' manualmente en tu navegador.")


if __name__ == "__main__":
    main()